In [ ]:
# Image Tagging with TensorFlow — Experiment Notebook
End-to-end CIFAR-10 CNN: load → augment → train → evaluate → predict.

In [ ]:
import os, sys
sys.path.append(os.path.abspath(".."))  # if notebook is inside notebooks/
import numpy as np, tensorflow as tf
import matplotlib.pyplot as plt
import config
from src.data_loader import load_cifar10, make_tf_datasets
from src.model import build_cnn_model, compile_model
from src.utils import set_seeds
print("TensorFlow:", tf.__version__)
config.ensure_dirs(); set_seeds()

In [ ]:
(x_train, y_train), (x_val, y_val), (x_test, y_test), class_names = load_cifar10()
print("Classes:", class_names)

In [ ]:
plt.figure(figsize=(10,4))
for i in range(10):
    plt.subplot(2,5,i+1)
    plt.imshow(x_train[i]); plt.axis("off")
    plt.title(class_names[y_train[i]])
plt.tight_layout(); plt.show()

In [ ]:
train_ds, val_ds = make_tf_datasets(x_train, y_train, x_val, y_val)
model = compile_model(build_cnn_model(include_augmentation=True))
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
cbs = [EarlyStopping(patience=8, restore_best_weights=True),
       ReduceLROnPlateau(factor=0.5, patience=3)]
history = model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=cbs)

In [ ]:
h = history.history
plt.plot(h["accuracy"], label="train"); plt.plot(h["val_accuracy"], label="val")
plt.legend(); plt.title("Accuracy"); plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
probs = model.predict(x_test); y_pred = probs.argmax(1)
print(classification_report(y_test, y_pred, target_names=class_names))

In [ ]:
model.save(config.MODEL_PATH)
print("Saved:", config.MODEL_PATH)